In [2]:
import torch
import utils
from dataset import get_data
from modules import ConstrainedHead

train_loader, val_w, test_w, stats = get_data("~/Desktop/LOBSTER")

cols = list(stats["mean"].index)
pos = [cols.index("spread")] + [i for i, c in enumerate(cols) if "gap" in c]
vol = [i for i, c in enumerate(cols) if "logv" in c]
head = ConstrainedHead(torch.tensor(stats["mean"].to_numpy()), torch.tensor(stats["std"].to_numpy()), pos, vol)

y = torch.randn(500, 24, 40) * 10                     # wild raw outputs, far outside the real range
x = utils.unnormalise(head(y), stats)
ix = utils.feature_index(stats)
print(f"Smallest spread / gap (ticks): {x[..., pos].min():.4f}   expect > 1")
print(f"Smallest log-volume: {x[..., vol].min():.4f}   expect >= 0")
print(f"Invariant report: {utils.invariant_rates(x, ix)}   expect 0 everywhere")

y0 = torch.randn(500, 24, 40)                         # typical z-score-sized outputs
print(f"Head changes typical values by at most: {(head(y0) - y0)[..., pos].abs().max():.3f} z   expect small (soft floor ≈ identity)")

Smallest spread / gap (ticks): 1.0000   expect > 1
Smallest log-volume: 0.0000   expect >= 0
Invariant report: {'Crossed books': 0.0, 'Ladder out of order': 0.0, 'Gap < 1 tick': 0.0, 'Negative volume': 0.0, 'Any invariant broken': 0.0}   expect 0 everywhere
Head changes typical values by at most: 4.232 z   expect small (soft floor ≈ identity)


In [3]:
train_x = train_loader.dataset.tensors[0]                        # real training windows (z-scores)
real_min = train_x.amin(dim=(0, 1))                               # lowest real z-score per feature
inside = y0 >= real_min                                           # values a real book could actually have
change = (head(y0) - y0).abs()
print(f"Values bent by more than 0.05 z: {(change[..., pos] > 0.05).float().mean()*100:.1f}% overall")
print(f"Within the real range: max change {change[..., pos][inside[..., pos]].max():.3f} z, "
      f"mean change {change[..., pos][inside[..., pos]].mean():.4f} z   expect small")

Values bent by more than 0.05 z: 85.2% overall
Within the real range: max change 0.605 z, mean change 0.1972 z   expect small
